<a href="https://colab.research.google.com/github/amr1311/Labor_Law_Chatbot/blob/main/Chat_bot_final__(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# connect with drive

In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


# library for pdf

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 9.6 MB/s eta 0:00:00


In [ ]:
!pip install pypdf sentence-transformers transformers numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 23.3 MB/s eta 0:00:00


# libaray for quantization of model

In [ ]:
!pip install bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


# read pdf and preprocessing for pdf

In [ ]:
from pypdf import PdfReader

def read_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    return text

In [ ]:
import re
from nltk.tokenize import sent_tokenize

# ============================================================
# 1) SPLIT INTO ARTICLES (ROBUST + SAFE)
# ============================================================
def split_by_articles(text):

    # 🔥 handle None
    if text is None:
        return []

    # 🔥 handle list / dict safely
    if isinstance(text, list):

        if len(text) > 0 and isinstance(text[0], dict):
            text = "\n".join(x.get("text", "") for x in text)
        else:
            text = "\n".join(map(str, text))

    elif isinstance(text, dict):
        text = text.get("text", "")

    # force string
    text = str(text)

    # normalize
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text).strip()

    # Article pattern
    pattern = r"(?m)^(Article\s+\d+\s*(?:[–\-:][^\n]*)?:?)"

    parts = re.split(pattern, text)

    articles = []

    if len(parts) > 0 and not re.match(r"Article\s+\d+", parts[0]):
        parts = parts[1:]

    for i in range(0, len(parts) - 1, 2):

        title = parts[i].strip()
        content = parts[i + 1].strip()

        if not title or len(content.split()) < 10:
            continue

        articles.append({
            "article": title,
            "text": content
        })

    return articles




In [ ]:
file_path = "/content/Labor-law.pdf"

# قراءة النص
text = read_pdf(file_path)

# تنظيف بسيط (مهم جدًا مع PDF)
text = text.strip()

# تقسيم
articles = split_by_articles(text)

# عرض النتائج
print("Total Articles:", len(articles))

# حماية من empty result
if len(articles) > 0:
    print("\nFIRST ARTICLE:\n")
    print("TITLE:", articles[0]["article"])
    print("TEXT (preview):")
    print(articles[0]["text"][:500])
else:
    print("❌ No articles found. Check PDF structure or regex.")

Total Articles: 311

FIRST ARTICLE:

TITLE: Article 1 – Promulgation:
TEXT (preview):
The provisions of this law and the accompanying law shall govern employment matters. 
They shall also apply, in the absence of specific provisions in individual employment contracts or 
collective labor agreements, to foreign workers within the Arab Republic of Egypt. 
Unless otherwise specifically provided, the provisions of this law and the accompanying law shall 
not apply to the following categories: 
• Employees of government bodies, including local administration units and public 
authorit


In [ ]:
articles


[{'article': 'Article 1 – Promulgation:',
  'text': 'The provisions of this law and the accompanying law shall govern employment matters. \nThey shall also apply, in the absence of specific provisions in individual employment contracts or \ncollective labor agreements, to foreign workers within the Arab Republic of Egypt. \nUnless otherwise specifically provided, the provisions of this law and the accompanying law shall \nnot apply to the following categories: \n• Employees of government bodies, including local administration units and public \nauthorities. \n• Domestic workers and those in similar categories.'},
 {'article': 'Article 2 – Promulgation:',
  'text': 'The Training and Qualification Fund, established under Labor Law No. 12 of 2003, shall retain its \nlegal public personality, report to the Minister responsible for labor affairs, and exercise its \npowers as regulated by the accompanying law. \n \nAll ongoing legal disputes—whether registered or pending before courts of all

In [ ]:
print("TITLE:", articles[13]["article"])

TITLE: Article 1:


In [ ]:
 print(articles[13]["text"][:2000])

For the purposes of applying the provisions of this Law, the following words and terms shall have 
the meanings assigned to each of them: 
Worker: Any natural person who works for a wage under the direction or supervision of an 
employer. 
Apprentice: Anyone engaged with an employer for the purpose of learning a trade, craft, or 
profession in return for a wage. 
Employer: Any natural or legal person employing one or more workers for a wage. 
Wage: Everything a worker receives in return for their work, whether in cash or in kind, 
including: 
• Basic Wage: The wage specified in the employment contract, along with any increases 
thereto. 
• Variable Wage: Other components of the wage, particularly: 
a. Commission or Percentage: A sum paid based on the worker’s production, 
sales, or collections. 
b. Bonuses: Monetary or percentage additions not included in the basic 
wage. 
c. Grants: Additional amounts given as customary or stipulated in individual 
or collective agreements. 
d. Reward

# Extract Article Number & Subtitle

In [ ]:
import re

def parse_article_title(title):

    match = re.match(r"Article\s+(\d+)\s*(?:[–\-]\s*(.*))?:?", title, re.IGNORECASE)

    if not match:
        return None, None

    number = int(match.group(1))
    subtitle = match.group(2) if match.group(2) else "General"

    return number, subtitle.strip()

# Clean & Structure Articles Data

In [ ]:
def clean_articles(articles):

    cleaned = []

    for a in articles:

        num, title = parse_article_title(a["article"])

        if num is None:
            continue

        cleaned.append({
            "id": num,
            "title": title,
            "text": a["text"],
            "raw": a["article"]
        })

    return cleaned

In [ ]:
articles=clean_articles(articles)

# Standardize Article Title Format
# Article X - Title

In [ ]:
def fix_title(article_id, raw_title):

    raw_title = raw_title.replace("Article", "").strip()

    if "–" in raw_title:
        parts = raw_title.split("–", 1)
        num = parts[0].strip()
        title = parts[1].strip()
    elif ":" in raw_title:
        parts = raw_title.split(":", 1)
        num = parts[0].strip()
        title = parts[1].strip()
    else:
        num = str(article_id)
        title = "General"

    return f"Article {num} - {title}"

In [ ]:
for a in articles:
    a["title"] = fix_title(a["id"], a["raw"])

In [ ]:
articles

[{'id': 1,
  'title': 'Article 1 - Promulgation:',
  'text': 'The provisions of this law and the accompanying law shall govern employment matters. \nThey shall also apply, in the absence of specific provisions in individual employment contracts or \ncollective labor agreements, to foreign workers within the Arab Republic of Egypt. \nUnless otherwise specifically provided, the provisions of this law and the accompanying law shall \nnot apply to the following categories: \n• Employees of government bodies, including local administration units and public \nauthorities. \n• Domestic workers and those in similar categories.',
  'raw': 'Article 1 – Promulgation:'},
 {'id': 2,
  'title': 'Article 2 - Promulgation:',
  'text': 'The Training and Qualification Fund, established under Labor Law No. 12 of 2003, shall retain its \nlegal public personality, report to the Minister responsible for labor affairs, and exercise its \npowers as regulated by the accompanying law. \n \nAll ongoing legal dis

In [ ]:
!pip install -U bitsandbytes accelerate

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import re
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# EMBEDDING MODEL
# =========================
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")


# =========================
# EMBEDDINGS
# =========================
def prepare_embeddings(articles):
    texts = [a["text"] for a in articles]
    embeddings = embedder.encode(texts, normalize_embeddings=True)
    return np.array(embeddings)


# =========================
# BETTER KEYWORD SCORE
# =========================
def keyword_score(query, text):

    query_tokens = re.findall(r"\w+", query.lower())
    text_tokens = re.findall(r"\w+", text.lower())

    if not text_tokens:
        return 0

    text_counter = Counter(text_tokens)

    score = 0
    for token in query_tokens:
        score += text_counter.get(token, 0)

    # normalize by text length (important fix)
    return score / (len(text_tokens) + 1e-9)


# =========================
# HYBRID RETRIEVE (IMPROVED)
# =========================
def retrieve(query, articles, embeddings, top_k=5, alpha=0.75):

    # ---- semantic (COSINE SIMILARITY) ----
    q_emb = embedder.encode([query], normalize_embeddings=True)

    semantic_scores = cosine_similarity(embeddings, q_emb).flatten()

    # normalize safely
    semantic_scores = (semantic_scores - semantic_scores.min()) / (
        semantic_scores.max() - semantic_scores.min() + 1e-9
    )

    # ---- keyword scores ----
    keyword_scores = np.array([
        keyword_score(query, a["text"]) for a in articles
    ])

    # normalize safely
    if keyword_scores.max() > 0:
        keyword_scores = keyword_scores / (keyword_scores.max() + 1e-9)

    # ---- hybrid fusion ----
    final_scores = alpha * semantic_scores + (1 - alpha) * keyword_scores

    # ---- top results ----
    top_idx = np.argsort(final_scores)[::-1][:top_k]

    results = []
    for i in top_idx:
        results.append({
            "id": articles[i]["id"],
            "title": articles[i]["title"],
            "text": articles[i]["text"],
            "score": float(final_scores[i])
        })

    return results

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# build embeddings once
embeddings = prepare_embeddings(articles)

query = "maternity leave rights"

results = retrieve(query, articles, embeddings, top_k=5)

for r in results:
    print(r["title"])
    print(r["text"][:300])
    print("score:", r["score"])
    print("-"*50)

Article 55 - 
After the maternity leave referred to in Article (54), a female worker has the right to return to 
her original job or a similar position, without losing any benefits that were applicable to her 
original role. 
 
It is prohibited to dismiss a female worker or terminate her employment during materni
score: 0.8983865275392644
--------------------------------------------------
Article 54 - 
A female worker is entitled to maternity leave for four months, including the period before and 
after delivery, provided the post-delivery leave is not less than 45 days. A medical certificate 
specifying the probable delivery date must be provided. This leave is paid. In all cases, a female 
worke
score: 0.86748120154047
--------------------------------------------------
Article 125 - 
The employer shall determine the timing of annual leave based on work requirements and 
conditions. Leave may not be interrupted except for compelling reasons required by the interest 
of work. 
 
Employ

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import re
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# MODELS
# =========================
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


# =========================
# EMBEDDINGS
# =========================
def prepare_embeddings(articles):
    texts = [a["text"] for a in articles]
    embeddings = embedder.encode(texts, normalize_embeddings=True)
    return np.array(embeddings)


# =========================
# KEYWORD SCORE
# =========================
def keyword_score(query, text):

    query_tokens = re.findall(r"\w+", query.lower())
    text_tokens = re.findall(r"\w+", text.lower())

    if not text_tokens:
        return 0

    text_counter = Counter(text_tokens)

    score = 0
    for token in query_tokens:
        score += text_counter.get(token, 0)

    return score / (len(text_tokens) + 1e-9)


# =========================
# HYBRID + RERANK
# =========================
def retrieve(query, articles, embeddings, top_k=5, alpha=0.75, rerank_top_k=10):

    # ===== 1) Semantic =====
    q_emb = embedder.encode([query], normalize_embeddings=True)
    semantic_scores = cosine_similarity(embeddings, q_emb).flatten()

    semantic_scores = (semantic_scores - semantic_scores.min()) / (
        semantic_scores.max() - semantic_scores.min() + 1e-9
    )

    # ===== 2) Keyword =====
    keyword_scores = np.array([
        keyword_score(query, a["text"]) for a in articles
    ])

    if keyword_scores.max() > 0:
        keyword_scores = keyword_scores / (keyword_scores.max() + 1e-9)

    # ===== 3) Hybrid Score =====
    hybrid_scores = alpha * semantic_scores + (1 - alpha) * keyword_scores

    # ناخد أفضل N للـ rerank
    top_idx = np.argsort(hybrid_scores)[::-1][:rerank_top_k]

    candidates = [articles[i] for i in top_idx]

    # ===== 4) RERANK (🔥 أهم خطوة)
    pairs = [(query, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)

    # ===== 5) ترتيب نهائي =====
    reranked = sorted(
        zip(candidates, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    results = []
    for doc, score in reranked:
        results.append({
            "id": doc["id"],
            "title": doc["title"],
            "text": doc["text"],
            "score": float(score)
        })

    return results


# =========================
# RUN
# =========================
embeddings = prepare_embeddings(articles)

query = "maternity leave rights"

results = retrieve(query, articles, embeddings)

for r in results:
    print(r["title"])
    print(r["text"][:300])
    print("score:", r["score"])
    print("-"*50)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

NameError: name 'articles' is not defined

In [ ]:
context = "\n\n".join([
    f"{r['title']}\n{r['text']}"
    for r in results
])

# installation for chromadb

In [ ]:
pip install chromadb

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# calling for saved file in chromdb

In [ ]:
client = chromadb.PersistentClient(path="/content/drive/MyDrive/chatbot_chroma")

In [ ]:
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
!pip install bitsandbytes accelerate transformers

In [ ]:
collection = client.get_or_create_collection(
    name="labor_law",
    embedding_function=embed_fn
)

# إنت دلوقتي محتاج تحوّل الملف النصي articles.txt إلى articles list قبل ما تضيفه لـ ChromaDB

In [ ]:
file_path = "/content/drive/MyDrive/articles/articles.txt"

with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

In [ ]:
import re

blocks = content.split("-"*50)

articles = []

for b in blocks:
    if not b.strip():
        continue

    id_match = re.search(r"ID:\s*(\d+)", b)
    title_match = re.search(r"Article:\s*(.*)", b)
    text_match = re.search(r"Text:\s*(.*)", b, re.S)

    if text_match:
        articles.append({
            "id": id_match.group(1) if id_match else "",
            "title": title_match.group(1) if title_match else "",
            "text": text_match.group(1).strip()
        })

# هذا الكود يستخدم لو معندكش ملف محفوظ فى الدرايف

In [ ]:
ids = []
documents = []
metadatas = []

for i, a in enumerate(articles):
    ids.append(str(i))
    documents.append(a["text"])
    metadatas.append({
        "title": a["title"]
    })

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

In [ ]:
print(client.list_collections())

[Collection(name=labor_law)]


In [ ]:
collection = client.get_collection(name="labor_law")

In [ ]:
collection.peek(5)

{'ids': ['0', '1', '2', '3', '4'],
 'embeddings': array([[ 0.02527432,  0.03692478, -0.02211472, ...,  0.00978682,
         -0.02546793,  0.03257611],
        [ 0.05578821, -0.04181442, -0.01721931, ..., -0.02012033,
         -0.06090729,  0.02091477],
        [ 0.04630245, -0.03443567, -0.01498929, ...,  0.00353127,
         -0.04380159,  0.0317603 ],
        [ 0.01707502, -0.00580142, -0.01055422, ..., -0.02687246,
         -0.06696154,  0.00511795],
        [ 0.03837884, -0.00840428,  0.05261393, ..., -0.00804842,
         -0.04924238,  0.00294172]]),
 'documents': ['The provisions of this law and the accompanying law shall govern employment matters. \nThey shall also apply, in the absence of specific provisions in individual employment contracts or \ncollective labor agreements, to foreign workers within the Arab Republic of Egypt. \nUnless otherwise specifically provided, the provisions of this law and the accompanying law shall \nnot apply to the following categories: \n• Employe

In [ ]:
query = "foreign workers in Egypt"

results = collection.query(
    query_texts=[query],
    n_results=5
)

for i, doc in enumerate(results["documents"][0]):
    print(i+1, doc[:200])

1 International organizations may carry out the placement of Egyptian workers with special skills 
and expertise for work abroad if the contract is with governmental bodies or Arab or foreign 
public au
2 The provisions of this law and the accompanying law shall govern employment matters. 
They shall also apply, in the absence of specific provisions in individual employment contracts or 
collective lab
3 Without prejudice to international agreements related to employment, the process of sending 
Egyptians to work domestically or abroad must be done through the relevant ministry or the 
following agenc
4 Shall be punished by imprisonment and a fine of not less than twenty thousand Egyptian pounds 
and not exceeding one hundred thousand Egyptian pounds, or by either of these two penalties, 
any person 
5 The relevant ministry, in cooperation with the ministries and relevant authorities, is responsible 
for monitoring the implementation of international agreements and contracts related to

In [ ]:
for i in range(5):
    print("ID:", data["ids"][i])
    print("TEXT:", data["documents"][i])
    print("META:", data["metadatas"][i])
    print("-"*50)

NameError: name 'data' is not defined

In [ ]:
query = "what are termination rights??"

results = collection.query(
    query_texts=[query],
    n_results=2
)

In [ ]:
print (results)

{'ids': [['21', '177']], 'embeddings': None, 'documents': [["The dissolution, liquidation, closure, or bankruptcy of the company does not prevent the \nfulfillment of all obligations arising under this law. \n \nA decision or ruling issued in this regard should specify a deadline for fulfilling the rights of \nemployees, and the relevant administrative authority must follow up on the fulfillment of these \nrights. It may act on behalf of the concerned parties to take necessary actions to meet these \nobligations within the specified deadline. \n \nThe competent minister will issue a decision specifying the controls, procedures, and deadlines \nfor fulfilling workers' rights.", "If the employer terminates an open-ended contract without a legitimate reason, the employee \nshall be entitled to compensation for damages, not less than two months' wages for each year of \nservice. This is without prejudice to the employee’s right to claim any other entitlements due \nunder the law. \n \nThe 

In [ ]:
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(meta["title"])
    print(doc[:1000])
    print("-"*70)

Article 9 - 
The dissolution, liquidation, closure, or bankruptcy of the company does not prevent the 
fulfillment of all obligations arising under this law. 
 
A decision or ruling issued in this regard should specify a deadline for fulfilling the rights of 
employees, and the relevant administrative authority must follow up on the fulfillment of these 
rights. It may act on behalf of the concerned parties to take necessary actions to meet these 
obligations within the specified deadline. 
 
The competent minister will issue a decision specifying the controls, procedures, and deadlines 
for fulfilling workers' rights.
----------------------------------------------------------------------
Article 165 - 
If the employer terminates an open-ended contract without a legitimate reason, the employee 
shall be entitled to compensation for damages, not less than two months' wages for each year of 
service. This is without prejudice to the employee’s right to claim any other entitlements due 
u

In [ ]:
!pip install -U bitsandbytes accelerate

# download from hugging face

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    local_dir="/content/drive/MyDrive/mistral",
    local_dir_use_symlinks=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

KeyboardInterrupt: 

# load model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from transformers import BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
model.save_pretrained("/content/drive/MyDrive/mistral")
tokenizer.save_pretrained("/content/drive/MyDrive/mistral")

In [ ]:
def generate_answer(query, retrieved_docs, model, tokenizer):

    context = "\n\n".join([
        f"{d['title']}\n{d['text']}" for d in retrieved_docs
    ])

    prompt = f"""
You are a legal question answering system.

Use ONLY the provided context to answer.

Rules:
- If the answer exists in the context, extract it clearly.
- If multiple parts exist, combine them simply.
- If partially missing, say what is available and mark missing parts.
- Never use external knowledge.
- Never change the topic.

Answer in simple Arabic using bullet points.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in full_output:
        answer = full_output.split("Answer:")[-1].strip()
    else:
        answer = full_output

    return answer

In [ ]:
query = "what are termination rights?"

# results is the dictionary returned by collection.query
# We need to transform it into the format expected by generate_answer

processed_results = []
num_results_to_take = 3
if results and "documents" in results and results["documents"] and \
   "metadatas" in results and results["metadatas"]:
    for i in range(min(num_results_to_take, len(results["documents"][0]))):
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        processed_results.append({
            "title": meta.get("title", "No Title"), # Use .get() for safety
            "text": doc
        })

answer = generate_answer(query, processed_results, model, tokenizer)

print(answer)

- Employee is entitled to compensation for damages if employer terminates an open-ended contract without a legitimate reason.
- Compensation is not less than two months' wages for each year of service.
- Termination for the following reasons is considered unjustified:
  - Employee's affiliation with a trade union or participation in union activities.
  - Serving or previously serving as a union representative or seeking such a role.
  - Filing a complaint or initiating legal action against the employer or participating in such actions regarding violations of law, regulations, or employment contracts.
  - Attachment of the employee's wages by court order.
  - Exercising the right to statutory leave.
  - Color, gender, marital status, family responsibilities, pregnancy, religion, or political opinion.


# evaluation

In [ ]:
def faithfulness_score(answer, retrieved_docs):

    context = " ".join([d["text"] for d in retrieved_docs]).lower()
    answer = answer.lower()

    words = answer.split()
    if len(words) == 0:
        return 0

    match = sum(1 for w in words if w in context)

    return match / len(words)

In [ ]:
score = faithfulness_score(answer, retrieved_docs)

print("Faithfulness:", score)

NameError: name 'retrieved_docs' is not defined

In [ ]:
def faithfulness_score_v2(answer, retrieved_docs):

    context = " ".join([d["text"] for d in retrieved_docs]).lower()
    answer = answer.lower()

    answer_sentences = answer.split(".")

    if len(answer_sentences) == 0:
        return 0

    scores = []

    for s in answer_sentences:
        s = s.strip()
        if not s:
            continue

        words = s.split()
        match = sum(1 for w in words if w in context)

        scores.append(match / max(len(words), 1))

    return sum(scores) / len(scores)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

def semantic_faithfulness(answer, retrieved_docs):

    context = " ".join([d["text"] for d in retrieved_docs])

    answer_emb = embedder.encode([answer])
    context_emb = embedder.encode([context])

    score = cosine_similarity(answer_emb, context_emb)[0][0]

    return float(score)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def rag_score(answer, retrieved_docs):

    f1 = faithfulness_score_v2(answer, retrieved_docs)
    f2 = semantic_faithfulness(answer, retrieved_docs)

    final = 0.5 * f1 + 0.5 * f2

    return {
        "faithfulness_word": f1,
        "faithfulness_semantic": f2,
        "final_score": final
    }

In [ ]:
retrieved_docs = processed_results
score = rag_score(answer, retrieved_docs)

print(score)

{'faithfulness_word': 0.9742063492063492, 'faithfulness_semantic': 0.8432645797729492, 'final_score': 0.9087354644896493}


In [ ]:
scores = []

queries = [
    "sick leave rights",
    "maternity leave",
    "termination rules",
    "working hours"
]

for query in queries:

    # retrieve
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=['documents', 'metadatas']
    )

    retrieved_docs = []
    for i in range(len(results["documents"][0])):
        retrieved_docs.append({
            "text": results["documents"][0][i],
            "title": results["metadatas"][0][i].get("title", "")
        })

    # generate answer
    answer = generate_answer(query, retrieved_docs, model, tokenizer)

    # evaluate
    score_dict = rag_score(answer, retrieved_docs)
    scores.append(score_dict["final_score"])

    print(f"Query: {query}")
    print(f"Score: {score_dict['final_score']}")
    print("-"*40)

# average
avg_score = sum(scores) / len(scores)

print("\nAverage Score:", avg_score)

Query: sick leave rights
Score: 0.9321436471186655
----------------------------------------
Query: maternity leave
Score: 0.9690229421737148
----------------------------------------
Query: termination rules
Score: 0.938805864687916
----------------------------------------
Query: working hours
Score: 0.9278504506991803
----------------------------------------

Average Score: 0.9419557261698691


# UI with streamlit

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 119.7 MB/s eta 0:00:00


In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.193.59:8501

Loading weights: 100% 291/291 [00:54<00:00,  5.33it/s, Materializing param=model.norm.weight] 
Loading weights: 100% 199/199 [00:00<00:00, 2522.39it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Stopping...
  Stopping...
Exception ignored in: <module 'threading' from '/usr/lib/python3.12/threading.py'>
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1624, in _shutdown
    lock.acquire()
  File "/usr/local/lib/python3.12/dist-packages/streamlit/web/bootstrap.py", line 43, in signal_handler
    server.stop()
  File "/usr/local/lib/python3.12/dist-packages/streamlit/web/server/server.py", line 550, in stop
    self._runtime.stop()
  File "/usr/local/lib/python3.12/dist-packages/streamlit/runtime/runtime.py", line 351, in stop
    async_objs.eventloop.call_soon_threadsafe(stop_on_eventloop)
  File "uvloop/loop.pyx", line 1290, in uvloop.loop.Loop.call_soon_threadsafe
  File "uvloop/loop.pyx", line 673, in uvloop.loop.Loop._append_ready_handle
  File "uvloop/loop.pyx", line 705, in uvloop.loop.Loop._check_closed
RuntimeError: Event loop is closed
terminate called without an active exception


In [ ]:
!pip install streamlit pyngrok

In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.168.78.228:8501

Loading weights: 100% 291/291 [00:59<00:00,  4.92it/s, Materializing param=model.norm.weight] 
Loading weights: 100% 199/199 [00:00<00:00, 859.69it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3CiqFLphdOQ8VjhyJ3PknDMTdDS_7AJs7m7gCsFgje6MMyymr")

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)

In [ ]:
with open("app.py", "r") as f:
    print(f.read())

In [ ]:
!cat app.py


import streamlit as st
import chromadb
from chromadb.utils import embedding_functions
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================
# Load Model (cached)
# =========================
@st.cache_resource
def load_model():
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        llm_int8_enable_fp32_cpu_offload=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    return tokenizer, model

tokenizer, model = load_model()

# =========================
# Load Chroma DB
# =========================
@st.cache_resource
def load_chroma():
    client = chromadb.PersistentCli

In [ ]:
%%writefile app.py

import streamlit as st
import chromadb
from chromadb.utils import embedding_functions
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================
# Load Model (cached)
# =========================
@st.cache_resource
def load_model():
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        llm_int8_enable_fp32_cpu_offload=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    return tokenizer, model

tokenizer, model = load_model()

# =========================
# Load Chroma DB
# =========================
@st.cache_resource
def load_chroma():
    client = chromadb.PersistentClient(
        path="/content/drive/MyDrive/chatbot_chroma"
    )

    embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="BAAI/bge-base-en-v1.5"
    )

    collection = client.get_collection(
        name="labor_law",
        embedding_function=embed_fn
    )

    return collection

collection = load_chroma()

# =========================
# LLM
# =========================
def generate_answer(query, retrieved_docs, model, tokenizer, language):

    context = "\n\n".join([
        f"{d['title']}\n{d['text']}" for d in retrieved_docs
    ])

    if language == "Arabic":
        instruction = """
You are a legal assistant.

Rules:
- Respond ONLY in Arabic
- Use only bullet points
- Maximum 5–7 points
- Keep answers simple and short
- Do not add extra explanations
"""
    else:
        instruction = """
You are a legal assistant.

Rules:
- Respond ONLY in English
- Use only bullet points
- Maximum 5–7 points
- Keep answers simple and short
- Do not add extra explanations
"""

    prompt = f"""
{instruction}

Rules:
- If the answer exists in the context, extract it clearly.
- If multiple parts exist, combine them simply.
- If partially missing, say what is available and mark missing parts.
- Never use external knowledge.
- Never change the topic..

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in full_output:
        answer = full_output.split("Answer:")[-1].strip()
    else:
        answer = full_output

    return answer

# =========================
# UI
# =========================

st.title("🤖 Labor Law Chatbot – Know Your Employee Rights ⚖️")

st.markdown("### 👷 Ask anything about your rights as an employee")

# اختيار اللغة
language = st.selectbox("Choose language / اختر اللغة", ["Arabic", "English"])

query = st.text_input("Ask your question / اكتب سؤالك:")

if st.button("Ask / اسأل"):
    if query:

        results = collection.query(
            query_texts=[query],
            n_results=3
        )

        docs = results["documents"][0]
        metas = results["metadatas"][0]

        retrieved_docs = []
        for doc, meta in zip(docs, metas):
            retrieved_docs.append({
                "title": meta.get("title", "No Title"),
                "text": doc
            })

        response = generate_answer(query, retrieved_docs, model, tokenizer, language)

        st.write("### Answer / الإجابة:")
        st.write(response)

Writing app.py


# extra code

In [ ]:
# ============================================================
# 2) SMART CHUNKING (SAFE)
# ============================================================
def chunk_text(text, chunk_size=5, overlap=1):

    if not text:
        return []

    sentences = sent_tokenize(str(text))
    chunks = []

    step = max(1, chunk_size - overlap)

    for i in range(0, len(sentences), step):

        chunk = " ".join(sentences[i:i + chunk_size]).strip()

        if len(chunk.split()) > 20:
            chunks.append(chunk)

    return chunks


# ============================================================
# 3) FULL PIPELINE
# ============================================================
def build_documents(text):

    articles = split_by_articles(text)

    docs = []

    for idx, art in enumerate(articles):

        chunks = chunk_text(art.get("text", ""))

        for j, ch in enumerate(chunks):

            docs.append({
                "id": f"art_{idx}_chunk_{j}",
                "article": art.get("article", ""),
                "text": ch
            })

    return docs

In [ ]:
import os

folder_path = "/content/drive/MyDrive/articles"
os.makedirs(folder_path, exist_ok=True)

In [ ]:
output_file_path = "/content/drive/MyDrive/articles/articles.txt"

with open(output_file_path, "w", encoding="utf-8") as f:
    for article in articles:

        # title موجود عندك
        title = article.get("title", "")

        # text موجود عندك
        text = article.get("text", "")

        # لو عايز id برضه
        doc_id = article.get("id", "")

        f.write(f"ID: {doc_id}\n")
        f.write(f"Article: {title}\n")
        f.write(f"Text: {text}\n")
        f.write("\n" + "-"*50 + "\n\n")

print("Saved successfully at:", output_file_path)

Saved successfully at: /content/drive/MyDrive/articles/articles.txt
